In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

import pandas as pd
import numpy as np
import pickle

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

train_df = pd.read_parquet(PROCESSED_DIR / "train.parquet")
test_df = pd.read_parquet(PROCESSED_DIR / "test.parquet")

with open(PROCESSED_DIR / "encoders.pkl", "rb") as f:
    encoders = pickle.load(f)

n_users = encoders["n_users"]
n_movies = encoders["n_movies"]

print(f"Train: {train_df.shape}, Test: {test_df.shape}")
train_df.head()

Train: (802553, 6), Test: (197656, 6)


,user_id,movie_id,rating,timestamp,user_idx,movie_idx
0,1,3186,4,978300019,0,2969
1,1,1270,5,978300055,0,1178
2,1,1721,4,978300055,0,1574
3,1,1022,5,978300055,0,957
4,1,2340,3,978300103,0,2147


In [2]:
global_mean = train_df["rating"].mean()
print(f"Global mean rating: {global_mean:.4f}")

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

# Predict global mean for every test row
test_df["pred_global_mean"] = global_mean

rmse_global = rmse(test_df["rating"], test_df["pred_global_mean"])
mae_global = mae(test_df["rating"], test_df["pred_global_mean"])

print(f"Global Mean Baseline -> RMSE: {rmse_global:.4f}, MAE: {mae_global:.4f}")

Global mean rating: 3.6172
Global Mean Baseline -> RMSE: 1.1478, MAE: 0.9447


In [3]:
movie_avg = train_df.groupby("movie_idx")["rating"].mean()

# Map to test set; fall back to global mean for movies unseen in train
test_df["pred_movie_avg"] = test_df["movie_idx"].map(movie_avg).fillna(global_mean)

rmse_movie_avg = rmse(test_df["rating"], test_df["pred_movie_avg"])
mae_movie_avg = mae(test_df["rating"], test_df["pred_movie_avg"])

print(f"Movie Average Baseline -> RMSE: {rmse_movie_avg:.4f}, MAE: {mae_movie_avg:.4f}")

n_unseen = test_df["movie_idx"].map(movie_avg).isna().sum()
print(f"Movies in test but not in train: {n_unseen}")

Movie Average Baseline -> RMSE: 0.9894, MAE: 0.7869
Movies in test but not in train: 54


In [4]:
# Popularity = number of ratings in train (could also weight by avg rating)
movie_popularity = train_df["movie_idx"].value_counts()
top_n_popular = movie_popularity.head(10).index.tolist()

print("Top 10 most popular movies (by idx):", top_n_popular)

def precision_recall_at_k(test_df, recommended_items, k=10, rating_threshold=4):
    """
    For each user, check how many of the top-k recommended items
    appear in their 'relevant' (rating >= threshold) test items.
    """
    relevant_per_user = (
        test_df[test_df["rating"] >= rating_threshold]
        .groupby("user_idx")["movie_idx"]
        .apply(set)
    )

    precisions, recalls = [], []
    rec_set = set(recommended_items[:k])

    for user_idx, relevant_items in relevant_per_user.items():
        if len(relevant_items) == 0:
            continue
        hits = len(rec_set & relevant_items)
        precisions.append(hits / k)
        recalls.append(hits / len(relevant_items))

    return np.mean(precisions), np.mean(recalls)

prec, rec = precision_recall_at_k(test_df, top_n_popular, k=10, rating_threshold=4)
print(f"Popularity Baseline -> Precision@10: {prec:.4f}, Recall@10: {rec:.4f}")


Top 10 most popular movies (by idx): [2651, 1106, 253, 1120, 1848, 575, 466, 2374, 1178, 1108]
Popularity Baseline -> Precision@10: 0.0320, Recall@10: 0.0258


In [5]:
baseline_results = {
    "global_mean": {"rmse": rmse_global, "mae": mae_global},
    "movie_average": {"rmse": rmse_movie_avg, "mae": mae_movie_avg},
    "popularity_topN": {"precision_at_10": prec, "recall_at_10": rec},
}

import json
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(exist_ok=True)

with open(REPORTS_DIR / "baseline_results.json", "w") as f:
    json.dump(baseline_results, f, indent=2)

print("Saved baseline_results.json")
baseline_results

Saved baseline_results.json


{'global_mean': {'rmse': np.float64(1.1477772815349552),
  'mae': np.float64(0.9446571839497945)},
 'movie_average': {'rmse': np.float64(0.9893960961603177),
  'mae': np.float64(0.7868903283172648)},
 'popularity_topN': {'precision_at_10': np.float64(0.03203492276695769),
  'recall_at_10': np.float64(0.025832298702690402)}}